# Diseño óptimo de perfiles por grupo (AISC 360-16 ASD + NCh2369:2025)

**Entradas que debes subir a Kaggle (dataset privado):**
1. El modelo SAP2000 con resultados (`.s2k` o `.txt`, el mismo que me pasaste comprimido).
2. `base_datos_perfiles.csv` (base de datos de perfiles validada).
3. (Opcional) `parametros_grupos.csv`: los parámetros por grupo que quieras fijar fuera del notebook. Si no lo subes, se usan los de la celda de configuración.

**Qué hace:** lee las barras de cada grupo y sus demandas ASD (32 combinaciones, todas las estaciones), evalúa cada perfil de la base de datos y entrega, por grupo, los perfiles que cumplen resistencia (compresión, tracción, flexión con pandeo lateral-torsional, corte e interacción H1), esbeltez local (NCh2369:2025 Tabla 9, con los vacíos completados desde la 2003) y esbeltez global (KL/r). Los ordena por peso total del grupo.

**Parámetros editables por grupo** (celda de configuración): tipo de miembro, familias permitidas, Kx, Ky, factor de longitud sin arriostrar, Cb, factor de utilización máximo, límite KL/r y excepción de 8.8.4 (0,7·R1 ≥ 1,0).

**Pendiente de confirmar antes de usar resultados:** Fy, Ry y E (valores de prueba en la celda de configuración), la aplicación de la excepción 8.8.4 por grupo y el criterio de Ry·Fy frente a Fy en la Tabla 9.


In [ ]:
import math, re, glob, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Materiales (ingresar) ----
FY_MPA = 248.2      # Fy [MPa]. Modelo: A36 = 2531 kgf/cm2 (= 248.2 MPa)  <-- CONFIRMAR
RY = 1.5            # Ry (factor de fluencia esperada)                     <-- CONFIRMAR
E_MPA = 200000.0    # E [MPa]
USE_RY = True       # True: Ry·Fy en Tabla 9 (conservador)
# ---- Diseño ----
DESIGN = 'ASD'      # el .s2k del modelo está en ASD (AISC 360-16)
UTIL_MAX = 1.0      # factor de utilización máximo por defecto (se puede cambiar por grupo)
LIM_KLR_DEFAULT = 200.0
# ---- Rutas ----
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working'
MODEL_FILE = None           # None = detectar .s2k o .txt con tablas SAP
BASE_DB = None              # None = buscar base_datos_perfiles.csv
# ---- Parámetros por grupo (editables). Una fila por grupo del modelo ----
GRUPOS = pd.DataFrame([
    # grupo, tipo, familias permitidas, Kx, Ky, Lb_factor, Cb, util_max, lim_KLr
    ['COLUMNAS', 'Columna', 'I soldada,I/H ala ancha,I,H,HSS cuadrado,HSS rectangular', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, LIM_KLR_DEFAULT, 'No'],
    ['VIGAS MARCOS', 'Viga', 'I soldada,I/H ala ancha,I,H', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, LIM_KLR_DEFAULT, 'No'],
    ['ARRIOSTRAMIENTO PISO', 'Arriostramiento', 'L (ángulo),HSS cuadrado,HSS rectangular,HSS circular', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, None, 'No'],
    ['ARRIOSTRAMIENTOS AX', 'Arriostramiento', 'L (ángulo),HSS cuadrado,HSS rectangular,HSS circular', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, None, 'No'],
    ['ARRIOSTRAMIENTOS AY', 'Arriostramiento', 'L (ángulo),HSS cuadrado,HSS rectangular,HSS circular', 1.0, 1.0, 1.0, 1.0, UTIL_MAX, None, 'No'],
], columns=['grupo', 'tipo_miembro', 'familias', 'Kx', 'Ky', 'Lb_factor', 'Cb', 'util_max', 'lim_KLr', 'excepcion_0_7R1'])
# Si existe parametros_grupos.csv en la entrada (editado y subido a Kaggle), reemplaza la tabla de arriba
_csv_grupos = glob.glob(f'{INPUT_DIR}/**/parametros_grupos.csv', recursive=True)
if _csv_grupos:
    GRUPOS = pd.read_csv(_csv_grupos[0])
    print('Parámetros de grupos leídos desde', _csv_grupos[0])
GRUPOS.to_csv(os.path.join(WORK_DIR, 'parametros_grupos_plantilla.csv'), index=False) if os.path.isdir(WORK_DIR) else None
print('Fy =', FY_MPA, 'MPa | Ry =', RY, '| RyFy =', round(RY * FY_MPA, 1), 'MPa | E =', E_MPA, 'MPa')

In [ ]:
KV = re.compile(r'(\w+)=("[^"]*"|\S+)')
def find_model():
    if MODEL_FILE: return MODEL_FILE
    hits = [p for p in glob.glob(f'{INPUT_DIR}/**/*', recursive=True) if p.lower().endswith(('.s2k', '.txt'))]
    hits = [p for p in hits if open(p, encoding='latin-1', errors='replace').read(4000).find('TABLE:') >= 0]
    if len(hits) != 1:
        raise FileNotFoundError(f'Se esperaba un modelo .s2k/.txt en {INPUT_DIR}; encontrados: {hits}. Fija MODEL_FILE.')
    return hits[0]

def logical_lines(path):
    """Une las líneas que SAP parte con ' _' al final."""
    buf = ''
    with open(path, encoding='latin-1', errors='replace') as f:
        for line in f:
            s = line.rstrip('\n')
            if s.rstrip().endswith('_'):
                buf += s.rstrip()[:-1] + ' '
                continue
            yield buf + s
            buf = ''

WANT = {'PROGRAM CONTROL', 'PREFERENCES - STEEL DESIGN - AISC 360-16', 'CONNECTIVITY - FRAME',
        'FRAME SECTION ASSIGNMENTS', 'GROUPS 2 - ASSIGNMENTS', 'ELEMENT FORCES - FRAMES'}
def read_model(path, grupos):
    out = {k: [] for k in WANT}
    cur = None
    gset = set(grupos)
    frames_in_groups = set()
    for ln in logical_lines(path):
        s = ln.strip()
        if s.startswith('TABLE:'):
            m = re.match(r'TABLE:\s+"(.+?)"', s)
            cur = m.group(1) if m else None
            continue
        if cur not in WANT or '=' not in s:
            continue
        if cur == 'GROUPS 2 - ASSIGNMENTS':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if d.get('ObjectType') == 'Frame' and d.get('GroupName') in gset:
                out[cur].append((d['GroupName'], d['ObjectLabel']))
        elif cur == 'ELEMENT FORCES - FRAMES':
            if 'OutputCase="ASD' not in s and 'OutputCase=ASD' not in s:
                continue
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append((d['Frame'], float(d['Station']), d['OutputCase'], float(d['P']), float(d['V2']),
                             float(d['V3']), float(d['M2']), float(d['M3'])))
        elif cur == 'PREFERENCES - STEEL DESIGN - AISC 360-16':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if 'OmegaB' in d: out[cur].append(d)
        elif cur == 'PROGRAM CONTROL':
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            if 'CurrUnits' in d: out[cur].append(d)
        else:
            d = {k: v.strip('"') for k, v in KV.findall(s)}
            out[cur].append(d)
    return out

model_path = find_model()
print('Modelo:', model_path, '|', round(os.path.getsize(model_path) / 1e6, 1), 'MB')
raw = read_model(model_path, GRUPOS['grupo'].tolist())
units = raw['PROGRAM CONTROL'][0]['CurrUnits'] if raw['PROGRAM CONTROL'] else '?'
print('Unidades del modelo:', units)
assert [p.strip().lower() for p in units.split(',')][:2] == ['kgf', 'm'], 'El modelo debe estar en kgf-m'
pref = raw['PREFERENCES - STEEL DESIGN - AISC 360-16'][0]
OMEGA_C = float(pref.get('OmegaC', 1.67)); OMEGA_B = float(pref.get('OmegaB', 1.67))
OMEGA_T = float(pref.get('OmegaTY', 1.67)); OMEGA_V = float(pref.get('OmegaV', 1.67))
OMEGA_V_ROLLED = float(pref.get('OmegaVRolledI', 1.5))
print(f'Ω del modelo: Ωc={OMEGA_C} Ωb={OMEGA_B} Ωt={OMEGA_T} Ωv={OMEGA_V} Ωv(I laminado)={OMEGA_V_ROLLED}')

In [ ]:
KGF = 9.80665
conn = pd.DataFrame(raw['CONNECTIVITY - FRAME'])[['Frame', 'Length']]
conn['L_mm'] = conn['Length'].astype(float) * 1000
sap_sec = pd.DataFrame(raw['FRAME SECTION ASSIGNMENTS'])[['Frame', 'AnalSect']]
fr = pd.DataFrame(raw['GROUPS 2 - ASSIGNMENTS'], columns=['grupo', 'Frame'])
fr = fr.merge(conn, on='Frame', how='left').merge(sap_sec, on='Frame', how='left')
fr = fr.merge(GRUPOS[['grupo', 'tipo_miembro', 'Kx', 'Ky', 'Lb_factor', 'Cb', 'util_max', 'lim_KLr', 'excepcion_0_7R1']], on='grupo')
missing = fr[fr['L_mm'].isna()]
assert missing.empty, f'Barras de grupo sin conectividad: {missing.Frame.tolist()}'
print('Barras en grupos:', len(fr), '| por grupo:', fr.groupby('grupo').size().to_dict())

dem = pd.DataFrame(raw['ELEMENT FORCES - FRAMES'], columns=['Frame', 'Station', 'combo', 'P', 'V2', 'V3', 'M2', 'M3'])
dem = dem[dem['Frame'].isin(fr['Frame'])].copy()
dem['P'] *= KGF; dem['V2'] *= KGF; dem['V3'] *= KGF
dem['M2'] *= KGF * 1000; dem['M3'] *= KGF * 1000          # kgf·m -> N·mm
dem = dem.merge(fr[['Frame', 'grupo']], on='Frame')
print('Filas de demanda ASD:', len(dem), '| combinaciones:', dem['combo'].nunique())
print(dem.groupby('grupo').agg(Pc_max_kN=('P', lambda s: round(-s.min()/1000, 1)),
                              Pt_max_kN=('P', lambda s: round(s.max()/1000, 1)),
                              M3_max_kNm=('M3', lambda s: round(s.abs().max()/1e6, 2)),
                              M2_max_kNm=('M2', lambda s: round(s.abs().max()/1e6, 2))))

In [ ]:
def find_db():
    if BASE_DB: return BASE_DB
    hits = glob.glob(f'{INPUT_DIR}/**/base_datos_perfiles.csv', recursive=True)
    assert len(hits) == 1, f'Se esperaba base_datos_perfiles.csv en {INPUT_DIR}; encontrados {hits}'
    return hits[0]
db_raw = pd.read_csv(find_db())
def fl(v):
    return float(v) if pd.notna(v) and v != '' else np.nan

def props(r):
    """Propiedades en mm, N. tipo_kind: I | RECT | ROUND | ANG | OTHER."""
    t = r['tipo']
    A = fl(r['A_cm2']) * 100
    kind = {'I soldada': 'I', 'I/H ala ancha': 'I', 'I': 'I', 'H': 'I', 'I inclinada': 'I',
            'HSS rectangular': 'RECT', 'HSS cuadrado': 'RECT', 'HSS circular': 'ROUND',
            'L (ángulo)': 'ANG'}.get(t, 'OTHER')
    d = dict(kind=kind, A=A, peso=fl(r['peso_kgf_m']))
    if kind == 'I':
        H, B, tw, tf = fl(r['H_mm']), fl(r['B_mm']), fl(r['tw_mm']), fl(r['tf_mm'])
        d.update(H=H, B=B, tw=tw, tf=tf,
                 Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4, Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3,
                 Zx=fl(r['Zx']) * 1e3, Zy=fl(r['Zy']) * 1e3, J=fl(r['J']) * 1e4 if pd.notna(r['J']) else np.nan)
        d['Cw'] = d['Iy'] * (H - tf) ** 2 / 4
    elif kind == 'RECT':
        H, B, t = fl(r['H_mm']), fl(r['B_mm']), fl(r['tw_mm'])
        d.update(H=H, B=B, tw=t, tf=t, Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4,
                 Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3)
        d['Zx'] = d['Sx'] * 1.12; d['Zy'] = d['Sy'] * 1.12        # aprox. para tubo rectangular (no viene en la planilla)
        Am = (B - t) * (H - t); p = 2 * ((B - t) + (H - t))
        d['J'] = 4 * Am ** 2 * t / p                               # torsión de sección cerrada de pared delgada
        d['Cw'] = 0.0
    elif kind == 'ROUND':
        D, t = fl(r['H_mm']), fl(r['tw_mm'])
        I = fl(r['Ix']) * 1e4
        d.update(H=D, B=D, tw=t, tf=t, Ix=I, Iy=I, Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wx']) * 1e3,
                 Zx=fl(r['Wx']) * 1e3 * 4 / 3 if pd.notna(r['Wx']) else np.nan)
        d['Zy'] = d['Zx']; d['J'] = 2 * I; d['Cw'] = 0.0
    elif kind == 'ANG':
        leg, t = fl(r['H_mm']), fl(r['tw_mm'])
        d.update(H=leg, B=fl(r['B_mm']), tw=t, tf=t, Ix=fl(r['Ix']) * 1e4, Iy=fl(r['Iy']) * 1e4,
                 Sx=fl(r['Wx']) * 1e3, Sy=fl(r['Wy']) * 1e3)
        d['rx'] = fl(r['ix']) * 10
        d['ry'] = fl(r['r_min_cm']) * 10 if pd.notna(r['r_min_cm']) and r['r_min_cm'] != '' else np.nan
        # Z plástico no viene en la planilla de ángulos: se aproxima como 1,5·S (flexión de ángulos no es el control)
        d['Zx'] = 1.5 * d['Sx']; d['Zy'] = 1.5 * d['Sy']
    return d

cat = db_raw[db_raw['tipo'].isin(['I soldada', 'I/H ala ancha', 'I', 'H', 'I inclinada', 'HSS rectangular',
                                   'HSS cuadrado', 'HSS circular', 'L (ángulo)'])].copy()
cat = cat[cat['validado'].astype(str).isin(['sí', 'si', 'True', 'true'])]
cat = cat[cat['A_cm2'].notna() & cat['Ix'].notna()]
PROPS = []
for _, r in cat.iterrows():
    d = props(r)
    if d['kind'] != 'ANG':
        d['rx'] = np.sqrt(d['Ix'] / d['A'])
        d['ry'] = np.sqrt(d['Iy'] / d['A'])
    d.update(designacion=r['designacion'], tipo=r['tipo'], familia=r['familia'],
             laminado=str(r['familia']).startswith('Copromet') or str(r['familia']).startswith('Cintac'))
    PROPS.append(d)
PROPS = [p for p in PROPS if not (p['kind'] == 'ANG' and not np.isfinite(p['ry']))]
cand = pd.DataFrame(PROPS)
print('Perfiles candidatos (validados, tipos soportados):', len(cand))
print(cand.groupby('tipo').size().to_dict())

In [ ]:
E = E_MPA
FYE = FY_EFF = RY * FY_MPA if USE_RY else FY_MPA
LAM_ALAS_NO_ATIESADA = 0.40 * math.sqrt(E / FYE)     # Tabla 9, alas no atiesadas I/H (2025)
LAM_ALMA_ARR = 1.57 * math.sqrt(E / FYE)              # Tabla 9, almas I/H/C arriostramiento (2025)
LAM_ALMA_FLEX_A = lambda ca: 3.96 * math.sqrt(E / FYE) * (1 - 3.04 * ca)  # Tabla 9, almas flexo-compresión, Ca<=0.114 (2025)
LAM_HSS_WALL = 1.40 * math.sqrt(E / FYE)              # 2003 Tabla 8.1 rectangular espesor uniforme (completa 2025)
LAM_ANG = 0.45 * math.sqrt(E / FYE)                   # 2003 Tabla 8.1 puntales de ángulos (completa)
LAM_GLOBAL_ARR = 1.5 * math.pi * math.sqrt(E / FY_MPA)  # NCh2369:2025 8.8.4 (usa Fy, no Ry·Fy)

def fcr_comp(lam, A):
    """Pn = Fcr·A según AISC E3 (lam = KL/r máx)."""
    Fe = math.pi ** 2 * E / lam ** 2
    if lam <= 4.71 * math.sqrt(E / FY_MPA):
        Fcr = 0.658 ** (FY_MPA / Fe) * FY_MPA
    else:
        Fcr = 0.877 * Fe
    return Fcr * A

def Mn_major(c, Lb, Cb):
    """Momento nominal eje fuerte (F2 para I doblemente simétricas; Mp en tubos y ángulos)."""
    Mp = FY_MPA * c['Zx']
    if c['kind'] != 'I' or not np.isfinite(c['J']):
        return Mp
    ry = c['ry']; Sx = c['Sx']; ho = c['H'] - c['tf']; J = c['J']
    Lp = 1.76 * ry * math.sqrt(E / FY_MPA)
    rts = math.sqrt(math.sqrt(c['Iy'] * c['Cw']) / Sx)
    Lr = 1.95 * rts * E / (0.7 * FY_MPA) * math.sqrt(J / (Sx * ho)) * \
         math.sqrt(1 + math.sqrt(1 + 6.76 * (0.7 * FY_MPA * Sx * ho / (E * J)) ** 2))
    if Lb <= Lp:
        return Mp
    if Lb <= Lr:
        return min(Cb * (Mp - (Mp - 0.7 * FY_MPA * Sx) * (Lb - Lp) / (Lr - Lp)), Mp)
    Fcr = Cb * math.pi ** 2 * E / (Lb / rts) ** 2 * math.sqrt(1 + 0.078 * J / (Sx * ho) * (Lb / rts) ** 2)
    return min(Fcr * Sx, Mp)

def Mn_minor(c):
    return min(FY_MPA * c['Zy'], 1.6 * FY_MPA * c['Sy'])

def Vn(c, axis):
    if c['kind'] == 'I':
        Aw = c['H'] * c['tw'] if axis == 3 else 2 * c['B'] * c['tf']
    elif c['kind'] == 'RECT':
        Aw = 2 * c['H'] * c['tw'] if axis == 3 else 2 * c['B'] * c['tw']
    elif c['kind'] == 'ROUND':
        Aw = 0.5 * c['A']
    else:
        return np.nan
    return 0.6 * FY_MPA * Aw

def check_section(c, frames_df, dem_g):
    """Devuelve (UC máximo, detalle) para un perfil frente a todas las barras de un grupo."""
    tipo = frames_df['tipo_miembro'].iloc[0]
    if c['kind'] == 'OTHER':
        return np.inf, {'motivo': 'tipo no soportado'}
    if c['kind'] == 'ANG' and not np.isfinite(c['ry']):
        return np.inf, {'motivo': 'sin r_min'}
    Ca_max = 0.0
    if c['kind'] == 'I' and tipo != 'Arriostramiento':
        Ca_max = max(0.0, OMEGA_C * (-dem_g['P'].min()) / (FYE * c['A']))
    uc_all, g_all = 0.0, {}
    for _, fr_ in frames_df.iterrows():
        dfr = dem_g[dem_g['Frame'] == fr_['Frame']]
        if dfr.empty:
            continue
        L = fr_['L_mm']
        kx, ky = fr_['Kx'] * L, fr_['Ky'] * L
        rx = c['rx']; ry = c['ry']
        lam = max(kx / rx, ky / ry)
        Pn = fcr_comp(lam, c['A'])
        Pc = Pn / OMEGA_C
        Tc = FY_MPA * c['A'] / OMEGA_T
        Mn3 = Mn_major(c, L * fr_['Lb_factor'], fr_['Cb'])
        Mn2 = Mn_minor(c)
        M3c, M2c = Mn3 / OMEGA_B, Mn2 / OMEGA_B
        V3c, V2c = Vn(c, 3), Vn(c, 2)
        omv3 = OMEGA_V_ROLLED if (c['laminado'] and c['kind'] == 'I' and c['tipo'] != 'I soldada') else OMEGA_V
        V3c = V3c / omv3 if np.isfinite(V3c) else np.inf
        V2c = V2c / OMEGA_V if np.isfinite(V2c) else np.inf
        P = dfr['P'].values
        comp = np.maximum(-P, 0)
        tens = np.maximum(P, 0)
        ratio_P = np.where(P < 0, comp / Pc, tens / Tc)
        cap = np.where(P < 0, Pc, Tc)
        m3 = np.abs(dfr['M3'].values) / M3c
        m2 = np.abs(dfr['M2'].values) / M2c
        ratio = np.where(ratio_P >= 0.2, ratio_P + 8 / 9 * (m3 + m2), ratio_P / 2 + (m3 + m2))
        vr3 = np.abs(dfr['V3'].values) / V3c if np.isfinite(V3c) else np.zeros(len(P))
        vr2 = np.abs(dfr['V2'].values) / V2c if np.isfinite(V2c) else np.zeros(len(P))
        ratio = np.maximum(ratio, np.maximum(vr3, vr2))
        if not np.all(np.isfinite(ratio)):
            return np.inf, {'motivo': 'resultado no finito'}
        uc = float(ratio.max())
        if uc > uc_all:
            uc_all = uc
            i = int(np.argmax(ratio))
            g_all = dict(frame=fr_['Frame'], combo=dfr['combo'].iloc[i], lam=round(lam, 1), UC=round(uc, 3))
        g_all.setdefault('lam_max', 0.0)
        g_all['lam_max'] = max(g_all['lam_max'], round(lam, 1))
    return uc_all, g_all

def check_local(c, tipo, Ca_max):
    """Relaciones ancho/espesor frente a los límites de NCh2369. Devuelve (ok, detalle)."""
    sq = math.sqrt(E / FYE)
    if c['kind'] == 'I':
        lam_f = (c['B'] / 2) / c['tf']
        lim_f = LAM_ALAS_NO_ATIESADA
        h_clear = c['H'] - 2 * c['tf']
        lam_w = h_clear / c['tw']
        if tipo == 'Arriostramiento':
            lim_w = LAM_ALMA_ARR
        else:
            lim_w = LAM_ALMA_FLEX_A(Ca_max) if Ca_max <= 0.114 else 1.57 * sq     # Ca>0.114: piso conservador
        ok = lam_f <= lim_f and lam_w <= lim_w
        return ok, dict(lam_ala=round(lam_f, 2), lim_ala=round(lim_f, 2), lam_alma=round(lam_w, 2), lim_alma=round(lim_w, 2))
    if c['kind'] == 'RECT':
        lam_b = (c['B'] - 2 * c['tw']) / c['tw']; lam_h = (c['H'] - 2 * c['tw']) / c['tw']
        ok = lam_b <= LAM_HSS_WALL and lam_h <= LAM_HSS_WALL
        return ok, dict(lam_ala=round(lam_b, 2), lim_ala=round(LAM_HSS_WALL, 2), lam_alma=round(lam_h, 2), lim_alma=round(LAM_HSS_WALL, 2))
    if c['kind'] == 'ROUND':
        lam = c['H'] / c['tw']
        lim = (0.062 if tipo == 'Arriostramiento' else 0.077) * E / FYE
        return lam <= lim, dict(lam_ala=round(lam, 2), lim_ala=round(lim, 2), lam_alma=None, lim_alma=None)
    if c['kind'] == 'ANG':
        lam = c['H'] / c['tw']
        return lam <= LAM_ANG, dict(lam_ala=round(lam, 2), lim_ala=round(LAM_ANG, 2), lam_alma=None, lim_alma=None)
    return False, {}

In [ ]:
RESULT_COLS = ['grupo', 'tipo_miembro', 'designacion', 'familia', 'tipo', 'peso_kgf_m', 'peso_total_kgf',
               'UC_max', 'util_max', 'lam_max', 'lim_lam', 'local_ok', 'global_ok', 'cumple', 'barra_critica',
               'combo_critica', 'detalle_local', 'sap_actual']
def run_group(g):
    """g: fila de GRUPOS. Evalúa todos los candidatos de sus familias y ordena por peso total."""
    fam_ok = [s.strip() for s in g['familias'].split(',')]
    frames_g = fr[fr['grupo'] == g['grupo']]
    dem_g = dem[dem['grupo'] == g['grupo']]
    c_pool = cand[cand['tipo'].isin(fam_ok)]
    lim_g = LAM_GLOBAL_ARR if g['tipo_miembro'] == 'Arriostramiento' else float(g['lim_KLr'])
    rows = []
    sap_list = ', '.join(sorted(frames_g['AnalSect'].dropna().unique()))
    for _, c in c_pool.iterrows():
        c = c.to_dict()
        uc, det = check_section(c, frames_g, dem_g)
        if not np.isfinite(uc):
            continue
        Ca_max = 0.0
        if c['kind'] == 'I' and g['tipo_miembro'] != 'Arriostramiento':
            Ca_max = max(0.0, OMEGA_C * (-dem_g['P'].min()) / (FYE * c['A']))
        ok_loc, det_loc = check_local(c, g['tipo_miembro'], Ca_max)
        lam_max = det.get('lam_max', 0.0)
        ok_glob = lam_max <= lim_g
        if str(g.get('excepcion_0_7R1', 'No')).strip().lower() in ('sí', 'si', 'yes', 'true'):
            ok_loc, ok_glob = True, True          # 8.8.4: exceptuado (0,7·R1 ≥ 1,0); el usuario debe confirmarlo
            det_loc = {'nota': 'excepción 8.8.4 (0,7R1≥1,0) aplicada'}
        peso_tot = float((frames_g['L_mm'] / 1000 * c['peso']).sum()) if np.isfinite(c['peso']) else np.nan
        cumple = (uc <= g['util_max']) and ok_loc and ok_glob
        rows.append(dict(grupo=g['grupo'], tipo_miembro=g['tipo_miembro'], designacion=c['designacion'],
                         familia=c['familia'], tipo=c['tipo'], peso_kgf_m=round(c['peso'], 2),
                         peso_total_kgf=round(peso_tot, 1), UC_max=round(uc, 3), util_max=g['util_max'],
                         lam_max=lam_max, lim_lam=round(lim_g, 1), local_ok=ok_loc, global_ok=ok_glob,
                         cumple=cumple, barra_critica=det.get('frame'), combo_critica=det.get('combo'),
                         detalle_local=str(det_loc), sap_actual=sap_list))
    res = pd.DataFrame(rows, columns=RESULT_COLS)
    return res.sort_values(['cumple', 'peso_total_kgf'], ascending=[False, True])

RESULTADOS = {}
for _, g in GRUPOS.iterrows():
    RESULTADOS[g['grupo']] = run_group(g)
    r = RESULTADOS[g['grupo']]
    ok = r[r['cumple']]
    print(f"\n=== {g['grupo']} ({g['tipo_miembro']}) | candidatos evaluados: {len(r)} | cumplen: {len(ok)} | util_max={g['util_max']}")
    print('    SAP actual:', r['sap_actual'].iloc[0] if len(r) else '-')
    if len(ok):
        print(ok.head(8)[['designacion', 'peso_kgf_m', 'peso_total_kgf', 'UC_max', 'lam_max', 'lim_lam', 'local_ok', 'global_ok', 'barra_critica', 'combo_critica']].to_string(index=False))
    else:
        print('    Ningún perfil del catálogo cumple. Motivo más frecuente (UC > util_max o esbeltez):')
        print(r[['designacion', 'UC_max', 'local_ok', 'global_ok']].head(5).to_string(index=False))

os.makedirs(WORK_DIR, exist_ok=True)
pd.concat(RESULTADOS.values()).to_csv(os.path.join(WORK_DIR, 'resultados_optimos_por_grupo.csv'), index=False)
print('\nResultados guardados en', os.path.join(WORK_DIR, 'resultados_optimos_por_grupo.csv'))

In [ ]:
# %% [interactivo] LISTA DESPLEGABLE DE PERFILES QUE CUMPLEN, POR GRUPO
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

def resumen_perfil(row):
    txt = [f"<b>{row['designacion']}</b> ({row['tipo']}) — peso {row['peso_kgf_m']} kgf/m, total grupo {row['peso_total_kgf']} kgf",
           f"UC máx = {row['UC_max']} (límite {row['util_max']}) · barra crítica {row['barra_critica']} · combo {row['combo_critica']}",
           f"Esbeltez global máx = {row['lam_max']} (límite {row['lim_lam']}) · local: {row['detalle_local']}"]
    return '<br>'.join(txt)

def tabla_cumplen(res):
    ok = res[res['cumple']]
    return ok[['designacion', 'tipo', 'peso_kgf_m', 'peso_total_kgf', 'UC_max', 'lam_max', 'lim_lam']]

PANELES = {}
for grupo, res in RESULTADOS.items():
    ok = res[res['cumple']].reset_index(drop=True)
    if ok.empty:
        display(HTML(f'<h4>{grupo}</h4><p>Ningún perfil del catálogo cumple con los parámetros actuales.</p>'))
        continue
    if HAS_WIDGETS:
        opciones = [(f"{r.designacion}  |  {r.peso_total_kgf} kgf  |  UC {r.UC_max}", i) for i, r in ok.iterrows()]
        dd = widgets.Dropdown(options=opciones, description='Perfil:', layout=widgets.Layout(width='700px'))
        salida = widgets.Output()
        def _mostrar(change, ok=ok, salida=salida):
            with salida:
                salida.clear_output()
                display(HTML(resumen_perfil(ok.loc[change['new']])))
        dd.observe(_mostrar, names='value')
        with salida:
            display(HTML(resumen_perfil(ok.loc[0])))
        PANELES[grupo] = widgets.VBox([widgets.HTML(f'<h4>{grupo}: {len(ok)} perfiles cumplen (mejor primero)</h4>'), dd, salida])
        display(PANELES[grupo])
    else:
        display(HTML(f'<h4>{grupo}: {len(ok)} perfiles cumplen</h4>'))
        display(tabla_cumplen(res))
